# Entraînement DPO — agent de triage CHSA (v5)

DPO par-dessus le **SFT v4** (Qwen3-1.7B-Base + LoRA, tokens de conversation entraînables), sur les **paires UltraMedical traduites et relues** (41 train / 4 validation).

**Inputs à attacher** (*Add Input*) :
1. le dataset contenant `train_dpo.py`, `train_dpo.jsonl`, `validation_dpo.jsonl` ;
2. la sortie (*Notebook Output*) du notebook SFT qui a produit **v4**.

Accélérateur : GPU T4. Exécution à valider par *Save Version > Save & Run All* (une session interactive non commitée est perdue).

## 1. Environnement

In [ ]:
# torchao est incompatible avec peft sur l'image Kaggle (piège déjà rencontré au SFT)
!pip uninstall -y torchao -q
import torch, transformers, peft, trl
print('torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| trl', trl.__version__)
print('GPU :', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 2. Chemins et contrôles des entrées
Arrêt immédiat si les données ou l'adaptateur ne sont pas les bons, plutôt qu'après un run inutile.

In [ ]:
import glob, json, os

dossier = glob.glob('/kaggle/input/**/train_dpo.py', recursive=True)[0].rsplit('/', 1)[0]

# Adaptateur v4 = tokens entraînables ET eval_loss ≈ 0,70 (écarte le dossier smoke_test du SFT,
# qui déclare aussi les tokens mais n'a pas d'évaluation)
def eval_loss(d):
    try:
        return json.load(open(os.path.join(d, 'all_results.json'))).get('eval_loss')
    except FileNotFoundError:
        return None

candidats = [p.rsplit('/', 1)[0] for p in glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
             if 'trainable_token_indices' in open(p).read()]
candidats = [d for d in candidats if eval_loss(d) is not None and abs(eval_loss(d) - 0.7006) < 0.01]
assert len(candidats) >= 1, "Adaptateur v4 introuvable : ajoute la sortie du notebook SFT v4 en input"
dossier_v4 = candidats[0]
res_v4 = {'eval_loss': eval_loss(dossier_v4)}
print('scripts/données :', dossier)
print('adaptateur v4   :', dossier_v4, '| eval_loss', res_v4['eval_loss'])
assert os.path.exists(os.path.join(dossier_v4, 'chat_template.jinja')), 'chat_template.jinja manquant dans le dossier v4'

for nom, attendu in [('train_dpo.jsonl', 41), ('validation_dpo.jsonl', 4)]:
    n = sum(1 for l in open(os.path.join(dossier, nom), encoding='utf-8') if l.strip())
    print(nom, n)
    assert n == attendu, f'{nom} : {n} lignes au lieu de {attendu} (mauvaise version des données ?)'

## 3. Smoke test (5 steps)
À vérifier dans la sortie : la ligne `[diag] norme du vecteur <|im_end|>` (deux valeurs **différentes**), `5/5` sans erreur, des logs `rewards/accuracies`.

`CUDA_VISIBLE_DEVICES=0` : un seul GPU, sinon Trainer active le parallélisme de données (constaté au SFT v4).

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python {dossier}/train_dpo.py \
    --adapter-sft-dir {dossier_v4} \
    --train-file {dossier}/train_dpo.jsonl \
    --val-file {dossier}/validation_dpo.jsonl \
    --output-dir /kaggle/working/smoke_test_dpo \
    --batch-size 1 --gradient-accumulation-steps 4 \
    --learning-rate 2e-5 --num-epochs 3 --save-steps 10 \
    --max-steps 5

## 4. Entraînement complet
41 paires, batch effectif 4, 3 époques : environ 30 steps.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python {dossier}/train_dpo.py \
    --adapter-sft-dir {dossier_v4} \
    --train-file {dossier}/train_dpo.jsonl \
    --val-file {dossier}/validation_dpo.jsonl \
    --output-dir /kaggle/working/v5-qwen3-1.7b-triage-dpo \
    --batch-size 1 --gradient-accumulation-steps 4 \
    --learning-rate 2e-5 --num-epochs 3 --save-steps 10

## 5. Contrôle du résultat
Un `!python` qui plante n'arrête pas le notebook : cette cellule fait échouer le commit si l'entraînement n'est pas allé au bout.

In [ ]:
sortie = '/kaggle/working/v5-qwen3-1.7b-triage-dpo'
res = json.load(open(os.path.join(sortie, 'all_results.json')))
for k in ['train_loss', 'eval_loss', 'eval_rewards/accuracies', 'eval_rewards/margins', 'train_runtime', 'epoch']:
    print(f'{k:28} {res.get(k)}')
assert res.get('epoch', 0) >= 3 and 'eval_loss' in res, 'Entraînement incomplet'

# Traçabilité : quel adaptateur SFT a servi de point de départ
json.dump({'adaptateur_sft_depart': dossier_v4, 'eval_loss_sft_v4': res_v4['eval_loss']},
          open(os.path.join(sortie, 'provenance_dpo.json'), 'w'), indent=2)

## 6. Archive à télécharger
Le zip s'appelle **v5** pour éviter toute confusion avec les archives précédentes.

In [ ]:
import shutil
shutil.rmtree('/kaggle/working/smoke_test_dpo', ignore_errors=True)
shutil.make_archive('/kaggle/working/v5-qwen3-1.7b-triage-dpo', 'zip', sortie)
print(os.listdir('/kaggle/working'))